[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/00b_matrix_quality_check.ipynb)

# R1-Q2 Week 1 (continued) — Matrix quality check

This notebook inspects the filtered shoot and root expression matrices
that `00_question_orientation.ipynb` produced. It does not change them,
re-filter them, or write anything to disk — it reads what's there and
reports back.

The point: a matrix can pass each per-step check in N0 (the log
transform worked, the MAD distribution looks right, the filtered shapes
are plausible) and still not be in shape for downstream co-expression
work. The diagnostics here are the next layer of "is this fit for
purpose" — sample-level structure, gene-level constancy, control
probes, and the whole-matrix correlation picture.

By the end of this notebook you will have:

- A per-condition sample count showing whether any tissue × stress cell
  is thinly populated.
- A sample-level PCA per tissue showing whether any samples sit far
  from the rest of their condition (outlier candidates).
- A pairwise sample-correlation heatmap per tissue showing whether
  replicates and conditions cluster as expected.
- A flag for genes that survived the MAD filter but have effectively
  zero range across samples.
- A count of control probes (`AFFX-*`, including the bacterial spike-in
  controls) that survived the filter, with a named decision for you to
  make.
- A four-number summary of the pairwise gene-gene correlation
  distribution per tissue, and a synthesis cell that names any concerns
  and points to next steps.

If the diagnostics come back clean, you proceed to Week 2's
`01_wgcna.ipynb`. If they surface a problem, you go back to N0, revise
the relevant pre-commit and re-run, then re-run this notebook to
confirm the fix. The synthesis at the end of this notebook is where
that branch decision gets made.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

#!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load and orient

Three pieces of input feed every diagnostic that follows:

1. **The two filtered parquets** (`shoot_filtered.parquet`,
   `root_filtered.parquet`) — what N0 wrote at the end of its Section
   3.
2. **The sample metadata** — which GSM belongs to which tissue, which
   stress condition, and which replicate. Comes from
   `iri.atgenexpress_metadata()`, the same function N0 used.
3. **`precommit.json`** — your Week 1 pre-commits. We don't act on
   these here, but we'll echo the `gene_filtering` block so the rest
   of the notebook has the filter choices visible on screen.

Load each in turn, then confirm the sample columns in the two
parquets line up with the metadata's GSM index. If any of these
loads fail, the error message tells you what to do.

In [ ]:
import pandas as pd

shoot_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_path = OUTPUT_DIR / "root_filtered.parquet"

try:
    shoot = pd.read_parquet(shoot_path)
    root = pd.read_parquet(root_path)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"\nCould not find a filtered matrix.\n"
        f"  Looked for: {shoot_path}\n"
        f"              {root_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its Section 3 writes both parquets to this location.\n"
    ) from None

print(f"Shoot matrix: {shoot.shape[0]:>5} genes × {shoot.shape[1]:>3} samples")
print(f"Root matrix:  {root.shape[0]:>5} genes × {root.shape[1]:>3} samples")

In [ ]:
# Same metadata function N0 used. The SOFT cache should be warm from
# N0's run; if it isn't, this will pull from GEO on first call.
metadata = iri.atgenexpress_metadata()

print(f"Metadata: {metadata.shape[0]} samples × {metadata.shape[1]} columns")
print(f"Columns:  {list(metadata.columns)}")
print(f"\nStress counts:")
print(metadata["stress"].value_counts().sort_index())
print(f"\nTissue counts:")
print(metadata["tissue"].value_counts())

In [ ]:
# Every sample column in the parquets should be a GSM the metadata
# knows about, and the tissue labels should match expectations.
shoot_gsms = set(shoot.columns)
root_gsms = set(root.columns)
metadata_gsms = set(metadata.index)

shoot_missing = shoot_gsms - metadata_gsms
root_missing = root_gsms - metadata_gsms

print(f"Shoot samples in metadata: {len(shoot_gsms - shoot_missing):>3} of {len(shoot_gsms)}")
print(f"Root samples in metadata:  {len(root_gsms - root_missing):>3} of {len(root_gsms)}")
if shoot_missing:
    print(f"  Shoot GSMs not in metadata: {sorted(shoot_missing)}")
if root_missing:
    print(f"  Root GSMs not in metadata:  {sorted(root_missing)}")

# Tissue label match: every shoot column should be metadata-tagged "shoot",
# every root column should be metadata-tagged "root".
shoot_tissues = metadata.loc[list(shoot_gsms & metadata_gsms), "tissue"].value_counts()
root_tissues = metadata.loc[list(root_gsms & metadata_gsms), "tissue"].value_counts()
print(f"\nShoot-matrix samples by tissue: {dict(shoot_tissues)}")
print(f"Root-matrix samples by tissue:  {dict(root_tissues)}")

assert not shoot_missing and not root_missing, "Some samples are not in metadata — investigate before continuing."
assert set(shoot_tissues.index) == {"shoot"}, "Shoot matrix contains non-shoot samples."
assert set(root_tissues.index) == {"root"}, "Root matrix contains non-root samples."
print("\nAlignment OK.")

### 2.4) Echo the pre-commit you're checking against

Read `precommit.json` and print the `gene_filtering` block. The
diagnostics that follow are checking whether the filter recorded here
actually produced matrices that are fit for WGCNA — so it helps to
have the filter choices on screen as you read the rest of the
notebook.

In [ ]:
import json

precommit_path = OUTPUT_DIR / "precommit.json"
try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find your pre-commit file.\n"
        f"  Expected at: {precommit_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes `precommit.json` to this location.\n"
    ) from None

gene_filtering = precommit.get("gene_filtering", {})
print("gene_filtering pre-commit:")
print(json.dumps(gene_filtering, indent=2))

You now have the two filtered matrices, the sample metadata, and your
gene-filtering pre-commit in scope. The next section looks at
sample-level structure: distribution per condition, PCA, and pairwise
correlation. That's the layer most likely to surface what was wrong
with the matrices Notebook 1 tried (and failed) to build a co-expression
network on.

## 3) Sample-level structure

Three diagnostics in this section, each looking at a different facet
of how samples relate to each other within a tissue:

- **Per-condition counts.** Is any tissue × stress cell thinly populated?
  A condition with very few samples can pass through the MAD filter
  without contributing meaningfully to gene-coexpression patterns —
  worth knowing before reading downstream module results.
- **Sample-level PCA.** Do the samples within a tissue cluster the way
  metadata suggests they should? Outlier samples — points sitting far
  from the rest of their condition — can degrade scale-free fit across
  an entire matrix without changing the median correlation distribution
  much. PCA is the easiest place to spot them.
- **Pairwise sample correlation heatmap.** Do replicates cluster
  together? Do related conditions show block structure? The
  sample × sample correlation matrix is a denser view of the same
  question the PCA scatter asks more loosely.

Run all three per tissue. Anything notable goes into the synthesis at
the end of the notebook.

### 3.1) Sample distribution per condition

A counts table showing how many samples each (tissue × stress) cell
contains. The total per tissue is fixed (124 each), so this surfaces
asymmetries — whether some stresses are over- or under-represented
relative to others, and whether shoot and root have the same coverage
per stress.

In [ ]:
import numpy as np

shoot_meta = metadata.loc[shoot.columns]
root_meta = metadata.loc[root.columns]

counts = pd.DataFrame({
    "shoot": shoot_meta["stress"].value_counts(),
    "root": root_meta["stress"].value_counts(),
}).sort_index().fillna(0).astype(int)
counts["both"] = counts["shoot"] + counts["root"]

print("Samples per (stress × tissue):")
print(counts)

cell_values = counts[["shoot", "root"]].values.flatten()
print(f"\nThinnest tissue × stress cell: {cell_values.min()} samples")
print(f"Thickest tissue × stress cell: {cell_values.max()} samples")
print(f"Median cell:                   {int(np.median(cell_values))} samples")

# Stresses where shoot and root coverage differ — usually they shouldn't.
asymmetric = counts[counts["shoot"] != counts["root"]]
if len(asymmetric):
    print(f"\nStresses with shoot ≠ root sample count:")
    print(asymmetric[["shoot", "root"]])
else:
    print("\nShoot and root sample counts match for every stress.")

### 3.2) Sample-level PCA per tissue

Run PCA on each tissue's filtered matrix (samples as observations,
genes as features). Plot the first two principal components as a
scatter, colored by stress condition. Print the proportion of variance
each of the top five PCs captures.

What to look at:

- **Cluster structure.** Do points of the same stress sit near each
  other? Replicates from the same stress at the same time point
  should cluster tightly. If they don't, sample-level noise is high.
- **Position of any single point.** Is any point conspicuously far
  from the rest of its color group? That's an outlier candidate —
  worth investigating before trusting downstream network-level
  patterns.
- **Variance captured by PC1.** If PC1 captures most of the variance
  (say > 60%), one axis is dominating the matrix. Could be a strong
  biological signal (e.g., one stress producing a much bigger response
  than the others) or could be a technical artifact (e.g., a batch
  effect aligned with one stress).

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def run_pca(matrix, n_components=5):
    """PCA on samples (transpose so rows are samples, columns are genes)."""
    pca = PCA(n_components=n_components)
    coords = pca.fit_transform(matrix.T.values)
    return pca, coords

shoot_pca, shoot_coords = run_pca(shoot)
root_pca, root_coords = run_pca(root)

# Set up a shared color map across both panels so the legend matches.
stresses = sorted(set(shoot_meta["stress"]) | set(root_meta["stress"]))
color_map = dict(zip(stresses, plt.cm.tab10(np.linspace(0, 1, len(stresses)))))

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(12, 5))

for stress in stresses:
    mask = (shoot_meta["stress"] == stress).values
    ax_shoot.scatter(shoot_coords[mask, 0], shoot_coords[mask, 1],
                     color=color_map[stress], label=stress, alpha=0.75, s=35)
    mask = (root_meta["stress"] == stress).values
    ax_root.scatter(root_coords[mask, 0], root_coords[mask, 1],
                    color=color_map[stress], label=stress, alpha=0.75, s=35)

ax_shoot.set_title(f"Shoot PCA (n={shoot.shape[1]} samples, {shoot.shape[0]} genes)")
ax_shoot.set_xlabel(f"PC1 ({shoot_pca.explained_variance_ratio_[0]:.1%} of variance)")
ax_shoot.set_ylabel(f"PC2 ({shoot_pca.explained_variance_ratio_[1]:.1%} of variance)")
ax_shoot.axhline(0, color="lightgrey", linewidth=0.5, zorder=0)
ax_shoot.axvline(0, color="lightgrey", linewidth=0.5, zorder=0)

ax_root.set_title(f"Root PCA (n={root.shape[1]} samples, {root.shape[0]} genes)")
ax_root.set_xlabel(f"PC1 ({root_pca.explained_variance_ratio_[0]:.1%} of variance)")
ax_root.set_ylabel(f"PC2 ({root_pca.explained_variance_ratio_[1]:.1%} of variance)")
ax_root.axhline(0, color="lightgrey", linewidth=0.5, zorder=0)
ax_root.axvline(0, color="lightgrey", linewidth=0.5, zorder=0)

handles, labels = ax_shoot.get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.08, 0.5),
           title="Stress", frameon=False)

plt.tight_layout()
plt.show()

print("Variance explained by PC (first five):")
print(f"  {'PC':>4}  {'shoot':>8}   {'root':>8}")
for i in range(5):
    print(f"  PC{i+1:<2}  {shoot_pca.explained_variance_ratio_[i]:>7.1%}   "
          f"{root_pca.explained_variance_ratio_[i]:>7.1%}")
print(f"  {'sum':>4}  {shoot_pca.explained_variance_ratio_[:5].sum():>7.1%}   "
      f"{root_pca.explained_variance_ratio_[:5].sum():>7.1%}")

### 3.3) Pairwise sample correlation heatmap per tissue

Compute the sample × sample Pearson correlation matrix within each
tissue, with samples ordered by stress, then by time point, then by
replicate. Plot as a heatmap, one per tissue.

What healthy structure looks like:

- **Bright diagonal** (correlation 1.0; each sample with itself).
- **Bright blocks along the diagonal** within each (stress, time) group
  — replicates from the same condition correlated near 1.0 with each
  other.
- **Slightly dimmer off-diagonal blocks** between different stresses —
  still high (these are all within-tissue samples), but not as high as
  within-condition.

Things that would be concerning:

- **A row/column noticeably darker than its neighbors.** That sample
  correlates poorly with everything, including its replicates. Outlier
  candidate — same role PCA plays in 3.2 from a different angle.
- **No visible block structure.** Suggests replicates aren't clustering
  with each other, which means either the metadata is wrong or the
  replicate variation is unusually high.

In [ ]:
def plot_sample_correlation(matrix, meta, title, ax):
    """Per-tissue sample correlation heatmap, ordered by (stress, time, replicate)."""
    ordered = meta.sort_values(["stress", "time_h", "replicate"])
    matrix_ordered = matrix[ordered.index]
    corr = matrix_ordered.corr()

    im = ax.imshow(corr.values, cmap="viridis", vmin=0.7, vmax=1.0, aspect="auto")
    ax.set_title(f"{title}  (vmin=0.7, vmax=1.0)")

    # Stress block boundaries and centered labels.
    stress_at_pos = ordered["stress"].values
    boundaries = []
    label_positions = []
    label_names = []
    start = 0
    for i in range(1, len(stress_at_pos) + 1):
        if i == len(stress_at_pos) or stress_at_pos[i] != stress_at_pos[i - 1]:
            label_positions.append((start + i - 1) / 2)
            label_names.append(stress_at_pos[i - 1])
            if i < len(stress_at_pos):
                boundaries.append(i)
            start = i

    for b in boundaries:
        ax.axhline(b - 0.5, color="white", linewidth=0.5)
        ax.axvline(b - 0.5, color="white", linewidth=0.5)

    ax.set_xticks(label_positions)
    ax.set_xticklabels(label_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(label_positions)
    ax.set_yticklabels(label_names, fontsize=8)
    return im

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(14, 6))

im_shoot = plot_sample_correlation(shoot, shoot_meta, "Shoot", ax_shoot)
im_root = plot_sample_correlation(root, root_meta, "Root", ax_root)

fig.colorbar(im_root, ax=[ax_shoot, ax_root], label="Pearson r", shrink=0.8)
plt.show()

# Per-tissue floor: the lowest off-diagonal correlation in the matrix.
def correlation_floor(matrix):
    corr = matrix.corr().values
    np.fill_diagonal(corr, np.nan)
    return np.nanmin(corr)

print(f"Lowest sample-sample correlation:")
print(f"  shoot:  {correlation_floor(shoot):.3f}")
print(f"  root:   {correlation_floor(root):.3f}")

Three views of sample-level structure are now on screen — per-condition
counts, PCA scatter, and pairwise correlation heatmap. Note anything
that looks off; the synthesis section at the end of the notebook is
where you'll consolidate those observations.

The next section drops down a level, from "are the samples in shape" to
"are the genes in shape." Specifically: did any genes survive the MAD
filter that don't actually vary in practice, and are there control
probes (the `AFFX-*` series) sitting in the matrix where biological
genes should be?

## 4) Gene-level structure

Section 3 looked at how samples relate to each other. This section
drops down a level to ask the same question about genes:

- **Did any genes survive the MAD filter that don't actually vary?**
  MAD is a *relative* variance measure — a gene can rank in the top
  25% by MAD while still having a tiny absolute range across samples.
  That's a known failure mode worth checking for explicitly.
- **Are control probes sitting in the matrix where biological genes
  should be?** The ATH1 array includes Affymetrix control probes
  (prefix `AFFX-*`) that aren't Arabidopsis genes at all. Some are
  bacterial spike-ins for normalizing labeling efficiency. If any
  survived the filter, that's a workflow risk to surface now rather
  than later.

Two subsections, both read-only against the in-memory matrices.

### 4.1) Effectively-constant gene check

For each gene, the *range* across samples (max minus min, on the log2
scale) is the absolute spread of expression — independent of MAD's
relative-to-median framing. A gene with range below ~1.0 log2 units
varies less than 2× end-to-end across every condition in the dataset.
For stress-response work, that's effectively flat.

Compute range per gene, plot the distribution, and flag any genes
below the threshold.

In [ ]:
def gene_range(matrix):
    """Per-gene range (max - min) across samples, on log2 scale."""
    return matrix.max(axis=1) - matrix.min(axis=1)

shoot_range = gene_range(shoot)
root_range = gene_range(root)

CONSTANT_THRESHOLD = 1.0  # log2 units; less than 2× end-to-end change

print("Per-gene range (log2 units):")
print(f"  shoot:  min={shoot_range.min():.3f}   median={shoot_range.median():.3f}   max={shoot_range.max():.3f}")
print(f"  root:   min={root_range.min():.3f}   median={root_range.median():.3f}   max={root_range.max():.3f}")

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

ax_shoot.hist(shoot_range, bins=80, color="steelblue")
ax_shoot.axvline(CONSTANT_THRESHOLD, color="black", linestyle="--", linewidth=1.5,
                 label=f"Threshold = {CONSTANT_THRESHOLD}")
ax_shoot.set_title(f"Shoot per-gene range (n={len(shoot_range)} genes)")
ax_shoot.set_xlabel("Range (log2 units)")
ax_shoot.set_ylabel("Gene count")
ax_shoot.legend()

ax_root.hist(root_range, bins=80, color="seagreen")
ax_root.axvline(CONSTANT_THRESHOLD, color="black", linestyle="--", linewidth=1.5,
                label=f"Threshold = {CONSTANT_THRESHOLD}")
ax_root.set_title(f"Root per-gene range (n={len(root_range)} genes)")
ax_root.set_xlabel("Range (log2 units)")
ax_root.legend()

plt.tight_layout()
plt.show()

shoot_constant = shoot_range[shoot_range < CONSTANT_THRESHOLD]
root_constant = root_range[root_range < CONSTANT_THRESHOLD]

print(f"\nGenes with range < {CONSTANT_THRESHOLD} log2 units (effectively constant):")
print(f"  shoot:  {len(shoot_constant)} of {len(shoot_range)} ({len(shoot_constant)/len(shoot_range):.1%})")
print(f"  root:   {len(root_constant)} of {len(root_range)} ({len(root_constant)/len(root_range):.1%})")

if len(shoot_constant):
    print(f"\nFlattest 5 shoot genes (smallest range):")
    print(shoot_range.sort_values().head().to_string())
if len(root_constant):
    print(f"\nFlattest 5 root genes (smallest range):")
    print(root_range.sort_values().head().to_string())

### 4.2) Control probe inspection

Affymetrix's ATH1 array includes a set of control probes prefixed
`AFFX-*`. Several categories exist on the array:

- **`AFFX-r2-Bs-*`** — *Bacillus subtilis* spike-in RNAs, added during
  sample preparation to normalize labeling and hybridization
  efficiency. Their signal should reflect technical variation across
  samples, not Arabidopsis biology.
- **`AFFX-r2-Ec-*`** — *E. coli* control transcripts. Same role,
  different organism.
- **`AFFX-r2-P1-*`** — phage P1 control transcripts.
- **Other `AFFX-*`** — Affymetrix's own array-QC probes (3'/5' bias
  controls, hybridization controls).

None of these are Arabidopsis genes. If any survived the MAD filter,
the concern is concrete:

- They can't form a biologically meaningful co-expression module.
- If one returns as a hub in Notebook 2, the result is
  uninterpretable — a *B. subtilis* spike-in is not an Arabidopsis
  stress-response gene.
- Their variance is most likely technical (hybridization batch
  effects), which means a module that forms around them tracks batch,
  not biology.

The decision this surfaces: should Notebook 0 filter out `AFFX-*`
probes explicitly, before the MAD ranking? If yes, that's a small N0
revision (one additional line in the filter step). If no — if you'd
rather catch them downstream when they matter — that's defensible,
but record it as a known risk in your EQ#1 writeup.

In [ ]:
shoot_affx = shoot.index[shoot.index.str.startswith("AFFX")]
root_affx = root.index[root.index.str.startswith("AFFX")]

print(f"Control probes (AFFX-*) surviving the MAD filter:")
print(f"  shoot:  {len(shoot_affx)} probes")
print(f"  root:   {len(root_affx)} probes")

def categorize(probes):
    """Bucket AFFX probes by spike-in category."""
    cats = {
        "Bs (B. subtilis spike-in)": 0,
        "Ec (E. coli spike-in)":     0,
        "P1 (phage spike-in)":       0,
        "legacy bacterial spike-in": 0,
        "other AFFX (array QC)":     0,
    }
    for p in probes:
        if "-Bs-" in p:
            cats["Bs (B. subtilis spike-in)"] += 1
        elif "-Ec-" in p:
            cats["Ec (E. coli spike-in)"] += 1
        elif "-P1-" in p:
            cats["P1 (phage spike-in)"] += 1
        # Add to the categorize function's match logic:
        elif any(legacy in p for legacy in ["-TrpnX-", "-LysX-", "-DapX-", "-ThrX-"]):
            cats["legacy bacterial spike-in"] += 1      
        else:
            cats["other AFFX (array QC)"] += 1
    return cats

if len(shoot_affx):
    print(f"\nShoot AFFX breakdown:")
    for k, v in categorize(shoot_affx).items():
        if v:
            print(f"  {k:32}  {v}")
if len(root_affx):
    print(f"\nRoot AFFX breakdown:")
    for k, v in categorize(root_affx).items():
        if v:
            print(f"  {k:32}  {v}")

# Per-probe ranges. A high range on an AFFX-* probe is more concerning
# than a low one — it means the probe is contributing real variance to
# any downstream module.
if len(shoot_affx) or len(root_affx):
    print(f"\nAFFX probes by per-gene range (log2 units):")
    if len(shoot_affx):
        print(f"\nShoot:")
        for probe in sorted(shoot_affx, key=lambda p: -shoot_range[p]):
            print(f"  {probe:32}  range={shoot_range[probe]:.2f}")
    if len(root_affx):
        print(f"\nRoot:")
        for probe in sorted(root_affx, key=lambda p: -root_range[p]):
            print(f"  {probe:32}  range={root_range[probe]:.2f}")
else:
    print("\nNo AFFX-* probes survived the filter in either tissue. Nothing to decide here.")

The gene-level findings — any effectively-constant genes, any control
probes — join the sample-level findings from Section 3 as inputs to
the synthesis at the end of the notebook. Note anything that warrants
a decision; the synthesis is where decisions get recorded.

## 5) Whole-matrix correlation distribution

Section 3 looked at samples, Section 4 looked at genes. This section
zooms back out to the matrix as a whole and asks a different
question: what does the *full distribution* of gene-gene pairwise
correlations look like?

WGCNA builds its co-expression network on top of this distribution.
The four-number summary below is a routine smoke test for matrix
quality. It does not replace the structural diagnostics in Sections 3
and 4 — a matrix can have a perfectly healthy summary here and still
have the dominant-axis structure Section 3's PCA surfaced. But the
numbers are worth recording for the writeup, and the contrast they
draw with Section 3's findings is itself instructive.

In [ ]:
def correlation_summary(matrix, sample_size=1_000_000, seed=0):
    """Four-number summary of gene-gene pairwise correlations.

    For matrices with thousands of genes, the full gene-gene correlation
    matrix is too large to materialize directly. Sample `sample_size`
    pairs at random and summarize.
    """
    rng = np.random.default_rng(seed)
    n_genes = matrix.shape[0]
    i_idx = rng.integers(0, n_genes, size=sample_size)
    j_idx = rng.integers(0, n_genes, size=sample_size)
    keep = i_idx != j_idx  # drop self-pairs
    i_idx, j_idx = i_idx[keep], j_idx[keep]

    # Per-gene z-scores → pairwise correlation = mean of element-wise product.
    means = matrix.values.mean(axis=1, keepdims=True)
    stds = matrix.values.std(axis=1, keepdims=True)
    z = (matrix.values - means) / stds
    pair_corrs = (z[i_idx] * z[j_idx]).mean(axis=1)

    abs_corrs = np.abs(pair_corrs)
    return {
        "median |r|":     float(np.median(abs_corrs)),
        "mean |r|":       float(np.mean(abs_corrs)),
        "95th pct |r|":   float(np.percentile(abs_corrs, 95)),
        "% pairs > 0.5":  float((abs_corrs > 0.5).mean() * 100),
    }

shoot_summary = correlation_summary(shoot)
root_summary = correlation_summary(root)

print("Gene-gene pairwise correlation summary (1M sampled pairs per tissue):")
print(f"  {'metric':18}  {'shoot':>10}   {'root':>10}")
print(f"  {'-'*18}  {'-'*10}   {'-'*10}")
for key in shoot_summary:
    print(f"  {key:18}  {shoot_summary[key]:>10.3f}   {root_summary[key]:>10.3f}")

The four numbers above are the matrix-quality smoke test. They join
the findings from Sections 3 and 4 as inputs to the synthesis.

A note on interpretation: if these summary numbers look reasonable
(median |r| in the 0.2–0.4 range, 95th percentile near 0.7, less than
~30% of pairs above 0.5) and yet WGCNA struggled on these matrices in
Notebook 1, that contrast is itself a finding. It says the global
correlation distribution is *not* a sufficient quality signal — you
need the structural diagnostics from Sections 3 and 4 to catch what's
actually going on. The synthesis section below makes that point
explicitly.

## 6) Synthesis

This section walks back through the diagnostics, names what they
collectively show, and records the next step for Week 2. For most
runs of this notebook on the AtGenExpress matrices, the next step
is "proceed to `01_wgcna.ipynb`." The synthesis is still worth
writing — it surfaces observations that belong in your EQ#1
writeup, and occasionally it surfaces a finding that does warrant
returning to N0 before continuing.

You'll write this section's interpretation yourself. The structure
below is a template — fill in the observations you actually made.
What you write here is the analytical core of your EQ#1.

### 6.1) What each diagnostic showed

Walk through each section and record what you observed. Be specific.

**Section 2 — Load and orient.**
- Filtered matrices loaded as expected: [shoot shape] and [root shape].
- Pre-commit gene-filtering parameters in scope: MAD top [X]% per
  tissue.
- Sample alignment with metadata: [confirmed / issue found].

**Section 3 — Sample-level structure.**
- *Per-condition counts.* [Symmetric or asymmetric across tissues?
  Any thinly-populated cells?]
- *PCA per tissue.* [Did points cluster by stress condition? Was any
  single condition driving most of the variance along PC1 or PC2?
  PC1 captured X% in shoot, Y% in root.]
- *Sample correlation heatmap.* [Did replicates within a condition
  correlate tightly? Was there any condition whose row/column showed
  conspicuously dark bands — i.e., correlating poorly with everything
  off-block?]
- *Correlation floor.* shoot [value], root [value].

**Section 4 — Gene-level structure.**
- *Effectively-constant genes.* [N] in shoot, [N] in root. Threshold
  was 1.0 log2 units.
- *AFFX control probes.* [N] surviving in shoot, [N] in root.
  Bacterial spike-in (Bs) count specifically: [N in shoot, N in
  root].

**Section 5 — Whole-matrix correlation distribution.**
- Median |r|: shoot [X], root [Y].
- 95th percentile |r|: shoot [X], root [Y].
- Percent of pairs above 0.5: shoot [X]%, root [Y]%.

### 6.2) Naming the finding

The single most important question for the writeup: **what does the
combined picture say about these matrices?**

Three patterns are possible, and the diagnostics distinguish among
them:

1. **Clean matrices, no structural concern.** Sample-level PCA shows
   per-condition clustering without any condition dragging PC1 far
   beyond the others. Heatmap shows balanced block structure. Gene-
   level diagnostics are clean (no effectively-constant genes; AFFX
   count is zero, as expected with N0's exclusion step). Correlation
   summary is in the healthy range. → Proceed to Week 2
   (`01_wgcna.ipynb`) under the original canonical hypothesis
   statement.

2. **Sample-level outliers driving the problem.** PCA shows one or a
   few specific points sitting far from their condition cluster.
   Correlation floor is conspicuously low in one tissue but not the
   other (e.g., < 0.5 in root, > 0.7 in shoot). → The matrices are
   recoverable. Identify the outlier GSMs, return to N0, and add a
   sample-removal step. Re-run the diagnostics here to confirm
   recovery before proceeding to Week 2.

3. **Condition-dominated variance structure.** PCA shows one or two
   stress conditions extending far along PC1 while the other
   conditions cluster more tightly. Correlation floors are similar
   between tissues. Whole-matrix correlation summary looks healthy
   in absolute terms. → The dataset's variance is structurally
   dominated by the strongest stress responses per tissue. This is
   a real biological feature of the AtGenExpress design — some
   stress collections drive more sample variance than others.
   Importantly, this observation does *not* prevent gene-level WGCNA
   from finding multi-module structure; it's a structural feature
   to flag for downstream interpretation rather than a problem to
   fix. → Proceed to Week 2 under the original canonical hypothesis
   statement, but record this observation for use in your EQ#2
   writeup.

Record which pattern your matrices show. Be specific about which
diagnostics support that read.

### 6.3) What this means for Week 2

Most patterns lead to the same next step: proceed to Week 2.

**If pattern 1 (clean):** Proceed to `01_wgcna.ipynb` under the
canonical hypothesis statement R1-Q2 was launched with. The
matrices are fit for what Week 2 expects.

**If pattern 2 (outliers):** Return to N0 and add a sample-removal
step. Re-run N0, then re-run this notebook. The branch decision is
made again on the new matrices.

**If pattern 3 (condition-dominated):** Proceed to `01_wgcna.ipynb`
under the canonical hypothesis statement R1-Q2 was launched with.
The condition-dominated variance you saw in Section 3 does not
prevent WGCNA from finding multi-module co-expression structure —
gene-level correlation structure is a different view of the data
than sample-level variance, and the two can show different things.

What you'll want to note in your EQ#2 writeup: modules whose
eigengenes correlate strongly with the dominant-variance stresses
will likely show higher correlation magnitudes than modules
tracking the other stresses. That isn't bias — it reflects how
variance is distributed across stresses in this dataset. When you
interpret your hubs in N2, account for this when comparing
correlation strengths across stresses; a hub in a
high-variance-stress module isn't automatically more biologically
important than one in a low-variance-stress module.

For students looking for an extension: a different filter design
might surface different network structure. The hypothesis: a filter
that selects for *multi-condition* variance (genes that respond
moderately across several stresses) rather than absolute variance
might surface different modules than the MAD filter does. Not
Week 1 scope; ambitious students can explore this in Week 3 or
later.

Whichever path you take, record the reasoning. That's the
analytical core of your EQ#1.

### 6.4) Anything to fix in N0?

N0's MAD filter is variance-based — it doesn't know about platform-
specific control probes. Any AFFX probes that surfaced in Section 4
are an expected feature of the ATH1 array, not a pipeline bug. The
decision §4.2 raised (drop AFFX before downstream analysis, or keep
them?) is yours to make. Dropping is the conservative choice —
spike-ins and array QC probes can't form a biologically meaningful
co-expression module, and an AFFX-prefixed hub returned by N2 would
be uninterpretable. If you decide to drop them, the simplest place
is here in 00b (a two-line filter on `shoot_filtered` and
`root_filtered` followed by overwriting the parquets); if you decide
to keep them, record the choice in EQ#1 and treat any AFFX-prefixed
hub from N2 as a known artifact rather than a finding.

One finding that can still warrant a small N0 revision:

- **A meaningful population of effectively-constant genes.** If
  Section 4 flagged more than ~1% of genes as effectively constant,
  the MAD filter is letting through too many flat genes. The fix is
  an additional `range > threshold` filter step in N0 — not a
  replacement for MAD, an additional filter on top of it. Re-run
  N0, then re-run this notebook to confirm.

Record any N0 revisions you made (or chose not to make) in your
EQ#1 writeup.

**End of Notebook 0b.** Your matrices have been characterized at
three levels (sample, gene, whole-matrix), and your observations
are recorded for EQ#1 and EQ#2.

What comes next depends on the branch decision in 6.3:

- **Pattern 1 or pattern 3:** open `01_wgcna.ipynb` and proceed to
  Week 2.
- **Pattern 2 (outliers):** return to `00_question_orientation.ipynb`,
  add the sample-removal step, re-run from there, then re-run this
  notebook to confirm before proceeding.

The canonical hypothesis statement on the Notion R1-Q2 page is the
one Week 2 operates under regardless of which pattern you saw —
patterns 1 and 3 both proceed without revising it.